# External Read Capabilities Demo

Explore the read-only capabilities of the LLM Chat Platform:
- **Web Read** — Fetch and parse web page content
- **Notion Read** — Fetch Notion page metadata

Both endpoints are separate from the main `/chat` write-path and provide stateless, read-only access to external resources.

## Setup

Configure the API base URL and load required libraries.

In [ ]:
import requests
import json
from datetime import datetime
from pprint import pprint
from typing import Dict, Any, Optional

# Configuration
API_BASE = "http://localhost:8000/"  # Change if API is running elsewhere
TIMEOUT = 15  # Request timeout in seconds

print(f"API Base URL: {API_BASE}")
print(f"Request Timeout: {TIMEOUT}s")
print()
print("✓ Libraries imported successfully")

## Utility Functions

Helper functions for making requests and displaying responses.

In [ ]:
def make_request(method: str, path: str, params: Optional[Dict[str, Any]] = None) -> Dict:
    """Make an HTTP request to the API."""
    url = f"{API_BASE}{path}"
    print(f"Request: {method} {path}")
    if params:
        print(f"Params: {params}")
    print()
    
    try:
        response = requests.request(
            method=method,
            url=url,
            params=params,
            timeout=TIMEOUT
        )
        return {
            "status_code": response.status_code,
            "headers": dict(response.headers),
            "data": response.json() if response.text else None,
            "text": response.text
        }
    except requests.exceptions.Timeout:
        return {
            "status_code": None,
            "error": f"Request timeout after {TIMEOUT}s"
        }
    except requests.exceptions.ConnectionError as e:
        return {
            "status_code": None,
            "error": f"Connection error: {e}"
        }
    except Exception as e:
        return {
            "status_code": None,
            "error": f"Unexpected error: {e}"
        }

def display_response(response: Dict) -> None:
    """Display a formatted response."""
    if "error" in response:
        print(f"❌ Error: {response['error']}")
    else:
        status = response["status_code"]
        status_emoji = "✓" if status == 200 else "⚠" if 400 <= status < 500 else "❌"
        print(f"{status_emoji} Status: {status}")
        print()
        print("Response Body:")
        pprint(response["data"])
    print()

print("✓ Utility functions defined")

## Web Read Examples

### Success Case: Fetch a Public Website

Read content from a public URL.

In [ ]:
# Example: Read example.com
response = make_request(
    "GET",
    "/web-read",
    params={"url": "https://matiasdasilva.com"}
)
display_response(response)

### Error Case: Validation Error (Missing Parameter)

What happens when a required parameter is missing?

In [ ]:
# Example: Missing 'url' parameter
response = make_request(
    "GET",
    "/web-read",
    params={}  # No 'url' parameter
)
display_response(response)

### Error Case: Network Error

What happens when the target URL is unreachable?

In [ ]:
# Example: Non-existent domain
response = make_request(
    "GET",
    "/web-read",
    params={"url": "https://this-domain-definitely-does-not-exist-12345.com"}
)
display_response(response)

## Notion Read Examples

### Setup: Configure Allowed Page IDs

Before testing Notion Read, ensure you have:
1. Set `NOTION_ROOT_PAGE_ID` environment variable
2. Added at least one page ID to `NOTION_ALLOWED_PAGE_IDS`
3. Restarted the application

To get a page ID from Notion:
- Open the page in Notion
- Copy the URL
- Extract the ID after the last `-` (format: `12345678-1234-5678-1234-567812345678`)

### Success Case: Fetch an Allowed Notion Page

Read metadata from an allowed Notion page.

In [ ]:
# Example: Replace with your actual allowed page ID
# Format: 12345678-1234-5678-1234-567812345678
ALLOWED_PAGE_ID = "12345678-1234-5678-1234-567812345678"  # Example placeholder

response = make_request(
    "GET",
    "/notion-read/page",
    params={"page_id": ALLOWED_PAGE_ID}
)
display_response(response)

### Error Case: Access Denied (Not in Allowlist)

What happens when you try to read a page that's not in the allowlist?

In [ ]:
# Example: Page ID not in allowlist
response = make_request(
    "GET",
    "/notion-read/page",
    params={"page_id": "12345678-1234-5678-1234-567812345678"}
)
display_response(response)

### Error Case: Validation Error (Missing Parameter)

What happens when the required `page_id` parameter is missing?

In [ ]:
# Example: Missing 'page_id' parameter
response = make_request(
    "GET",
    "/notion-read/page",
    params={}  # No 'page_id' parameter
)
display_response(response)

## Error Handling & Status Codes

### Common Status Codes

| Code | Meaning | Example Cause |
|------|---------|---------------|
| **200** | Success | Valid request, data returned |
| **403** | Forbidden | Page ID not in allowlist (Notion) or URL blocked (Web) |
| **422** | Validation Error | Missing or empty required parameter |
| **502** | Bad Gateway | Network error or upstream service failure |
| **504** | Gateway Timeout | Upstream request timeout (Notion only) |
| **503** | Service Unavailable | MCP subprocess unavailable (Notion only) |
| **500** | Server Error | Unexpected server error |

For detailed troubleshooting, see `docs/troubleshooting_external_read.md`.

## Response Schema

### WebReadOut (Web Read)

```json
{
  "url": "string (original request URL)",
  "final_url": "string (after redirects)",
  "content_type": "string (MIME type)",
  "title": "string (page title)",
  "text": "string (plain text content)",
  "truncated": "boolean (content truncated?)"
}
```

### NotionPageOut (Notion Read)

```json
{
  "page_id": "string (UUID format)",
  "title": "string (page title)",
  "url": "string (Notion page URL)",
  "created_time": "string (ISO 8601 timestamp)",
  "last_edited_time": "string (ISO 8601 timestamp)"
}
```

## Configuration & Verification

### Health Check: Verify Endpoints are Reachable

In [ ]:
# Check if endpoints are reachable
print("Checking endpoint reachability...")
print()

# Web Read
try:
    response = requests.get(f"{API_BASE}/web-read", timeout=5)
    if response.status_code in [200, 422]:  # 200 = success, 422 = validation error (expected)
        print("✓ Web Read endpoint is reachable")
    else:
        print(f"⚠ Web Read endpoint returned status {response.status_code}")
except Exception as e:
    print(f"❌ Web Read endpoint is NOT reachable: {e}")

print()

# Notion Read
try:
    response = requests.get(f"{API_BASE}/notion-read/page", timeout=5)
    if response.status_code in [200, 403, 422, 503]:  # Expected status codes
        print("✓ Notion Read endpoint is reachable")
    else:
        print(f"⚠ Notion Read endpoint returned status {response.status_code}")
except Exception as e:
    print(f"❌ Notion Read endpoint is NOT reachable: {e}")

print()
print("Endpoint verification complete.")

## Key Points

### About These Endpoints

- **Stateless:** Each request is independent; no session or state is maintained
- **Read-Only:** No changes are made to Notion or external resources
- **Separate from /chat:** These do not invoke the LLM or write to the conversation database
- **Metadata-Only:** Notion Read returns page metadata, not full page content
- **Structured Logging:** All requests are logged with `request_id` for tracing

### Configuration

For production use, configure:

**Web Read:**
- `WEBREAD_BLOCKED_DOMAINS` — Domains to block
- `WEBREAD_TIMEOUT_SECONDS` — Fetch timeout
- `WEBREAD_MAX_CONTENT_LENGTH` — Max content size

**Notion Read:**
- `NOTION_ROOT_PAGE_ID` — Root page for Notion workspace
- `NOTION_ALLOWED_PAGE_IDS` — Allowed page IDs (JSON array)
- `NOTION_MCP_SERVER_COMMAND` — MCP server command
- `NOTION_MCP_TIMEOUT_SECONDS` — MCP request timeout

### Documentation

- **Endpoint Guide:** `docs/external_read_capabilities.md`
- **Troubleshooting:** `docs/troubleshooting_external_read.md`
- **Error Reference:** `docs/error_decision_table.md`